## **Project**
Weddings requires a huge amount of planning from venue, catering, transportation, music, etc... It is a perfect usecase for a multi-agent architecture

- Build the following sub-agents:
    - A travel agent that finds flights to and from your ideal destination
    - A venue agent that search the web for wedding venue
    - A DJ agent that scours the music database for a playlist matching the right genre
- Build a coordinator who puts the vision together


### **Architecture**
1. **Sub-Agent 1**
    - Name: Travel Agent
    - Tool: kiwi mcp server
    - Input: Origin and Destination
2. **Sub-Agent 2**
    - Name: Venue Agent
    - Tool: Web Search
    - Input: Destination and Guest Count
3. **Sub-Agent 3**
    - Name: DJ Agent
    - Tool: SQL Query Tool to query the music database
    - Input: Genre
4. **Coordinator Agent Job**
    - Figure out the key details about the wedding (like origin, destination, guest count and music genre) and update the state with them
    - Call on the sub-agents to do their predefined tasks
    - Finally compile the results from each sub-agent and return to the user

In [1]:
import os

# Setup LANGSMITH API Key
f = open('keys/.langsmith_api_key.txt')
LANGSMITH_API_KEY = f.read()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = "my-simple-chain-project"

In [2]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
                 model="gpt-4o-mini",
                 temperature=0.0)

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "my_travel_server": {
          "transport": "streamable_http",
          "url": "https://mcp.kiwi.com"
        }
    }
)

flight_toolset = await client.get_tools()

print(f"Loaded {len(flight_toolset)} MCP Tools: {[tool.name for tool in flight_toolset]}")

Loaded 2 MCP Tools: ['search-flight', 'feedback-to-devs']


In [4]:
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from typing import Dict, Any

search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """ Search the web for information """
    return search_tool.invoke(query)

In [5]:
# import sqlite3

# try:
#     con = sqlite3.connect('database/Chinook.db')
#     cursor = con.cursor()
#     cursor.execute("""
#     SELECT name 
#     FROM sqlite_master 
#     WHERE type='table';
#     """)
    
#     tables = cursor.fetchall()
    
#     print(tables)
    
# finally:
#     con.close()
#     print('DONE!!')

In [6]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///database/Chinook.db")

db.run("select * from genre")

"[(1, 'Rock'), (2, 'Jazz'), (3, 'Metal'), (4, 'Alternative & Punk'), (5, 'Rock And Roll'), (6, 'Blues'), (7, 'Latin'), (8, 'Reggae'), (9, 'Pop'), (10, 'Soundtrack'), (11, 'Bossa Nova'), (12, 'Easy Listening'), (13, 'Heavy Metal'), (14, 'R&B/Soul'), (15, 'Electronica/Dance'), (16, 'World'), (17, 'Hip Hop/Rap'), (18, 'Science Fiction'), (19, 'TV Shows'), (20, 'Sci Fi & Fantasy'), (21, 'Drama'), (22, 'Comedy'), (23, 'Alternative'), (24, 'Classical'), (25, 'Opera')]"

In [7]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

db = SQLDatabase.from_uri("sqlite:///database/Chinook.db")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_toolset = toolkit.get_tools()

print(f"Loaded {len(sql_toolset)} MCP Tools: {[tool.name for tool in sql_toolset]}")

Loaded 4 MCP Tools: ['sql_db_query', 'sql_db_schema', 'sql_db_list_tables', 'sql_db_query_checker']


## **Create a State**

In [8]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: int
    genre: str

## **Create Sub-Agent 1**

In [9]:
from langchain.agents import create_agent

# Travel agent
travel_agent = create_agent(
    model=llm,
    tools=flight_toolset,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [10]:
# Venue agent
venue_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    """
)

In [13]:
response = venue_agent.invoke({"messages" : "marriage in london"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

marriage in london
================================== Ai Message ==================================
Tool Calls:
  web_search (call_xFgRGGo63pqFQ7Y6KZiQvOsW)
 Call ID: call_xFgRGGo63pqFQ7Y6KZiQvOsW
  Args:
    query: wedding venues in London
================================= Tool Message =================================
Name: web_search

December 5, 2025 -London offers unforgettable venues with waterfront views, convenient accommodations & unique spaces for your dream celebration. Find your perfect wedding venue! March 26, 2025 -The choices, however, are seemingly endless so to help you out we’ve picked London’s top luxury wedding venues, from Richmond to Covent Garden, Holborn to Marylebone and South Kensington to Greenwich. Whether you want a big showstopper wedding in one of the capital’s most glamorous hotels, a laid-back bohemian setting to say ‘I do’, a ballroom fit for royalty, or nuptials in the g

In [14]:
# Playlist agent
playlist_agent = create_agent(
    model=llm,
    tools=sql_toolset,
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.
    You may need to make multiple queries to iteratively find the best options.
    """
)

In [15]:
response = playlist_agent.invoke({"messages" : "jazz"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

jazz
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_AYczukVkGkyFaCURDaXFWqX2)
 Call ID: call_AYczukVkGkyFaCURDaXFWqX2
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_8cMuuIlxYgXIm3F9Do2nK92Y)
 Call ID: call_8cMuuIlxYgXIm3F9Do2nK92Y
  Args:
    table_names: Track, Genre, Playlist, PlaylistTrack
  sql_db_schema (call_5B3k57xKnVheY0YOmaiO7J3W)
 Call ID: call_5B3k57xKnVheY0YOmaiO7J3W
  Args:
    table_names: Album, Artist
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Genre" (
	"GenreId

## **Main Coordinator**

In [20]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state.get("origin")
    destination = runtime.state.get("destination")
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    # print(f" [Searching flights] From {origin} to {destination}")
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state.get("destination")
    capacity = runtime.state.get("guest_count")
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state.get("genre")
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre"""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )

In [21]:
from langchain.agents import create_agent

coordinator = create_agent(
    model=llm,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding planning coordinator with 4 subagents: update state, venue, flights and playlist.
    
    Analyze the user request to parse the important state arguments like origin, destination, guest count, and genre.
    Make sure to delegate to ONE subagent at a time.
    After each subagent response, summarize and check if planning is complete.
    When all major aspects (venue, catering, flowers) are planned, respond with FINAL ANSWER.
    
    Do not loop endlessly. Finish after 3-4 delegations max.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)

## **Test**

In [22]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from new delhi and I'd like a wedding in paris for 100 guests, jazz genre")],
    }, 
    config={"recursion_limit": 100}
)

In [23]:
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

I'm from new delhi and I'd like a wedding in paris for 100 guests, jazz genre
================================== Ai Message ==================================
Tool Calls:
  update_state (call_fNQr8bcF0oOtvqJ2Mxow4i1K)
 Call ID: call_fNQr8bcF0oOtvqJ2Mxow4i1K
  Args:
    origin: new delhi
    destination: paris
    guest_count: 100
    genre: jazz
================================= Tool Message =================================
Name: update_state

Successfully updated state
================================== Ai Message ==================================
Tool Calls:
  search_venues (call_8DxA7FykfSiPxtaAGVFzjh29)
 Call ID: call_8DxA7FykfSiPxtaAGVFzjh29
  Args:
================================= Tool Message =================================
Name: search_venues

Here are some wedding venue options in Paris for 100 guests, focusing on affordability, capacity, and reviews:

1. **Le Pavillon Wagram**
   - **Capaci

## **Important Observations**
1. By default the agent will have a hard recursion limit of 25 steps. Beyond this recursion_limit it will throw `GraphRecursionError`. It can be handled by changing the recursion_limit in the config argument.
2. Agent can fail to plan if a good quality reasoning LLM is not provided.
```
================================ Human Message =================================

I'm from new delhi and I'd like a wedding in paris for 100 guests, jazz genre
================================== Ai Message ==================================
Tool Calls:
  search_venues (call_r68Zzq5Zry2Q5qHa6hZe0dub)
 Call ID: call_r68Zzq5Zry2Q5qHa6hZe0dub
  Args:
================================= Tool Message =================================
Name: search_venues

I was unable to find specific wedding venues based on the criteria of "None" for location and "None" for guests. However, I can provide general information about affordable wedding venues for small weddings with high ratings. Here are some options that are often recommended:

1. **Community Centers**: Many community centers offer affordable rental rates and can accommodate small gatherings. They often have basic amenities and can be decorated to fit your theme.

2. **Backyards**: Hosting your wedding in a backyard can be a cost-effective option. It allows for a personal touch and can be decorated to create a beautiful setting.

3. **Local Parks**: Many parks have picnic areas or pavilions that can be rented for a small fee. This option provides a natural backdrop and is often budget-friendly.

4. **Small Restaurants or Cafes**: Some restaurants offer private dining rooms for small events. This can be a great way to have a cozy wedding with good food.

5. **Barns or Farmhouses**: Rustic venues like barns or farmhouses can provide a charming atmosphere for a small wedding. They often have beautiful outdoor spaces as well.

6. **Art Galleries or Museums**: Some galleries and museums offer rental spaces for events. These venues can provide a unique and artistic backdrop for your wedding.

7. **Airbnb or Vacation Rentals**: Renting a large Airbnb can be a great way to host a small wedding. Many properties have beautiful settings and can accommodate your guests comfortably.

8. **Church Halls**: Many churches have halls that can be rented for events. They are often affordable and can be decorated to fit your wedding theme.

For the best options, consider looking for venues that have good reviews and are known for their affordability. You can also check local listings or wedding planning websites for specific venues in your area that meet your criteria.
================================== Ai Message ==================================
Tool Calls:
  search_flights (call_hIEm9NL0A9o4gxK968klg6Vm)
 Call ID: call_hIEm9NL0A9o4gxK968klg6Vm
  Args:
================================= Tool Message =================================
Name: search_flights

It seems that the origin and destination for the flight search are not specified. Please provide the departure city or airport and the destination city or airport for me to assist you in finding the best flight options.
```